# 標本空間と事象 — インタラクティブデモ

このノートブックでは、確率論の出発点となる**標本空間・事象・事象の演算**を、
具体的な試行（コイン・サイコロ・カード）と可視化を通じて体験的に理解します。

---
## 目次
1. ライブラリのインポート
2. 標本空間の構成と可視化
   - 2-1. コイン投げ（1回・2回・3回）
   - 2-2. サイコロ（1個・2個）
   - 2-3. 標本空間のサイズの爆発（べき集合）
3. 事象の定義と部分集合
4. 事象の演算
   - 4-1. 和事象・積事象・補事象
   - 4-2. ベン図による可視化
   - 4-3. 排反事象と完全加法系
5. 直積空間と多段試行
6. 連続標本空間（区間・平面）
7. 事象の独立性と排反性の違い

## 1. ライブラリのインポート

In [ ]:
# ── 必要なパッケージのインストール ──────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'matplotlib-venn', 'japanize-matplotlib'], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle, Circle, Ellipse
from matplotlib.colors import to_rgba
import matplotlib.patheffects as pe
import japanize_matplotlib
from matplotlib_venn import venn2, venn3
from itertools import product
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
print('ライブラリの読み込み完了 ✓')

---
## 2. 標本空間の構成と可視化

**標本空間**（Sample Space）$\Omega$ とは、ある試行（experiment）で起こりうる
すべての結果（outcome）の集合です。

> $\Omega = \{\omega_1, \omega_2, \ldots\}$

標本空間の要素 $\omega \in \Omega$ を**根元事象**（elementary event / sample point）と呼びます。

### 2-1. コイン投げ（1回・2回・3回）

In [ ]:
# コインの面
coin = ['H', 'T']  # Head / Tail

# n 回投げた場合の標本空間を生成
def coin_space(n):
    return list(product(coin, repeat=n))

spaces = {n: coin_space(n) for n in [1, 2, 3]}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

colors_hh = {'H': '#4C72B0', 'T': '#DD8452'}

for ax, (n, sp) in zip(axes, spaces.items()):
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, len(sp) - 0.5)
    ax.set_title(f'コイン {n} 回投げ\n|Ω| = {len(sp)}', fontsize=12)
    ax.axis('off')

    # 外枠（標本空間を表す長方形）
    rect = Rectangle((-0.4, -0.4), 1.8, len(sp) - 0.2,
                     linewidth=2, edgecolor='#555', facecolor='#f8f8f8')
    ax.add_patch(rect)
    ax.text(1.3, len(sp) - 0.3, 'Ω', fontsize=14, color='#555', ha='right')

    for i, outcome in enumerate(sp):
        label = ' '.join(outcome)
        # 各コインの色分け
        x_offset = 0.0
        for c in outcome:
            circ = Circle((x_offset + 0.1, len(sp) - 1 - i),
                          0.18, color=colors_hh[c], zorder=3)
            ax.add_patch(circ)
            ax.text(x_offset + 0.1, len(sp) - 1 - i, c,
                    ha='center', va='center', fontsize=9,
                    color='white', fontweight='bold', zorder=4)
            x_offset += 0.45

# 凡例
axes[0].add_patch(Circle((-0.5, -0.5), 0, color='#4C72B0', label='H (表)'))
axes[0].add_patch(Circle((-0.5, -0.5), 0, color='#DD8452', label='T (裏)'))
fig.legend(['H (表)', 'T (裏)'],
           handler_map={str: mpatches.Patch},
           handles=[mpatches.Patch(color='#4C72B0', label='H (表)'),
                    mpatches.Patch(color='#DD8452', label='T (裏)')],
           loc='lower center', ncol=2, fontsize=11, frameon=False)

fig.suptitle('コイン投げの標本空間（根元事象を列挙）', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

for n, sp in spaces.items():
    print(f'コイン {n} 回: Ω = {{", ".join([str(o) for o in sp])}}  |Ω| = {len(sp)}')

### 2-2. サイコロ（1個・2個）

2個のサイコロを投げる試行では、標本空間は**直積空間**になります：

$$\Omega = \{1,2,3,4,5,6\} \times \{1,2,3,4,5,6\}$$

$|\Omega| = 6 \times 6 = 36$

In [ ]:
# 2個のサイコロの全根元事象をグリッドで可視化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：1個のサイコロ
ax = axes[0]
omega1 = list(range(1, 7))
colors1 = plt.cm.Blues(np.linspace(0.35, 0.8, 6))
for i, face in enumerate(omega1):
    circ = Circle((0.5, i), 0.38, color=colors1[i], zorder=2)
    ax.add_patch(circ)
    ax.text(0.5, i, str(face), ha='center', va='center',
            fontsize=16, fontweight='bold', color='white', zorder=3)
rect = Rectangle((-0.1, -0.5), 1.2, 6.0,
                 linewidth=2, edgecolor='#555', facecolor='#f8f8f8', zorder=1)
ax.add_patch(rect)
ax.text(0.9, 5.5, 'Ω', fontsize=14, color='#555')
ax.set_xlim(-0.2, 1.1)
ax.set_ylim(-0.6, 6.0)
ax.set_title('サイコロ 1 個\n|Ω| = 6', fontsize=12)
ax.axis('off')

# 右：2個のサイコロ（グリッド）
ax2 = axes[1]
omega2 = list(product(range(1, 7), repeat=2))
colors2 = plt.cm.YlOrRd(np.linspace(0.2, 0.8, 36))

for idx, (d1, d2) in enumerate(omega2):
    row = d2 - 1   # y軸：2個目
    col = d1 - 1   # x軸：1個目
    c = plt.cm.Blues(0.4 + (d1 + d2) / 24)
    circ = Circle((col, row), 0.38, color=c, zorder=2)
    ax2.add_patch(circ)
    ax2.text(col, row, f'{d1},{d2}', ha='center', va='center',
             fontsize=7.5, color='white', fontweight='bold', zorder=3)

rect2 = Rectangle((-0.5, -0.5), 6, 6,
                  linewidth=2, edgecolor='#555', facecolor='#f0f4f8', zorder=1)
ax2.add_patch(rect2)
ax2.set_xlim(-0.6, 5.8)
ax2.set_ylim(-0.6, 5.8)
ax2.set_xticks(range(6))
ax2.set_xticklabels([f'1個目={i}' for i in range(1, 7)], fontsize=8, rotation=30)
ax2.set_yticks(range(6))
ax2.set_yticklabels([f'2個目={i}' for i in range(1, 7)], fontsize=8)
ax2.set_title('サイコロ 2 個（直積空間）\n|Ω| = 36', fontsize=12)
ax2.text(5.5, 5.5, 'Ω', fontsize=14, color='#555', ha='right')
ax2.spines['top'].set_visible(True)
ax2.spines['right'].set_visible(True)

fig.suptitle('サイコロの標本空間', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'サイコロ 1 個: |Ω| = {len(omega1)}')
print(f'サイコロ 2 個: |Ω| = {len(omega2)}  （= 6 × 6）')

### 2-3. 標本空間のサイズの爆発（べき集合）

**べき集合**（Power Set）$2^\Omega$ とは、$\Omega$ のすべての部分集合の集合であり、
事象全体を表します。$|\Omega| = n$ のとき $|2^\Omega| = 2^n$。

コインを $n$ 回投げる試行では $|\Omega| = 2^n$ となり、事象の数は $2^{2^n}$ と急激に増加します。

In [ ]:
ns = np.arange(1, 9)
omega_sizes    = 2 ** ns                    # |Ω|
powerset_sizes = 2 ** omega_sizes.astype(float)  # |2^Ω|

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：|Ω| の増加
axes[0].bar(ns, omega_sizes, color='steelblue', edgecolor='white')
axes[0].set_xticks(ns)
axes[0].set_xticklabels([f'n={i}' for i in ns])
axes[0].set_ylabel('根元事象の数 |Ω|', fontsize=11)
axes[0].set_title('コインの投げ回数 n と |Ω| = 2^n', fontsize=12)
for i, v in zip(ns, omega_sizes):
    axes[0].text(i, v + 0.5, f'2^{i}={v}', ha='center', fontsize=8)

# 右：|2^Ω| の超爆発（対数スケール）
valid = powerset_sizes < 1e300
axes[1].semilogy(ns[valid], powerset_sizes[valid], 'o-',
                 color='#C44E52', linewidth=2, markersize=8)
axes[1].set_xticks(ns[valid])
axes[1].set_xticklabels([f'n={i}' for i in ns[valid]])
axes[1].set_ylabel('事象の総数 |2^Ω|（対数スケール）', fontsize=11)
axes[1].set_title('事象の数 = 2^|Ω| の超指数的増加', fontsize=12)
for i, v in zip(ns[valid], powerset_sizes[valid]):
    axes[1].text(i, v * 1.5, f'{int(v):,}', ha='center', fontsize=8, rotation=20)

plt.tight_layout()
plt.show()

print('n回コイン投げ：')
for n in range(1, 6):
    sz = 2**n
    print(f'  n={n}: |Ω|={sz:4d}  事象の数 |2^Ω| = 2^{sz} = {2**sz:,}')

---
## 3. 事象の定義と部分集合

**事象**（Event）$A$ とは、標本空間 $\Omega$ の部分集合 $A \subseteq \Omega$ です。

| 事象の種類 | 定義 | 例（サイコロ）|
|-----------|------|------|
| 根元事象 | $\{\omega\}$（1要素の集合） | $\{3\}$：3の目 |
| 複合事象 | 複数の根元事象の集合 | $\{2,4,6\}$：偶数の目 |
| 全事象 | $\Omega$ | $\{1,2,3,4,5,6\}$ |
| 空事象 | $\emptyset$ | 7以上の目（起こりえない）|

以下では2個のサイコロ（$|\Omega|=36$）を例に、様々な事象を可視化します。

In [ ]:
omega2 = [(d1, d2) for d1 in range(1, 7) for d2 in range(1, 7)]

# 事象の定義
events_def = {
    'A: 和が 7\n{(d1,d2) | d1+d2=7}':  lambda d1, d2: d1 + d2 == 7,
    'B: 両方が偶数\n{(d1,d2) | d1,d2∈{2,4,6}}': lambda d1, d2: d1 % 2 == 0 and d2 % 2 == 0,
    'C: 1個目が 3 以下\n{(d1,d2) | d1<=3}':      lambda d1, d2: d1 <= 3,
    'D: ゾロ目\n{(d1,d2) | d1=d2}':             lambda d1, d2: d1 == d2,
}

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, (name, cond) in zip(axes, events_def.items()):
    event_set = [(d1, d2) for d1, d2 in omega2 if cond(d1, d2)]
    event_mask = np.array([cond(d1, d2) for d1, d2 in omega2])

    for (d1, d2), in_event in zip(omega2, event_mask):
        color = '#4C72B0' if in_event else '#DDDDDD'
        circ = Circle((d1 - 1, d2 - 1), 0.38,
                      color=color, zorder=2)
        ax.add_patch(circ)
        ax.text(d1 - 1, d2 - 1, f'{d1},{d2}',
                ha='center', va='center', fontsize=6.5,
                color='white' if in_event else '#888',
                fontweight='bold' if in_event else 'normal', zorder=3)

    ax.set_xlim(-0.6, 5.6)
    ax.set_ylim(-0.6, 5.6)
    ax.set_xticks(range(6))
    ax.set_xticklabels(range(1, 7), fontsize=8)
    ax.set_yticks(range(6))
    ax.set_yticklabels(range(1, 7), fontsize=8)
    ax.set_title(f'{name}\n|A|={len(event_set)}', fontsize=9)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

    # P(A)の表示
    ax.text(2.5, -0.4, f'P = {len(event_set)}/36 = {len(event_set)/36:.3f}',
            ha='center', fontsize=9, color='#333')

fig.suptitle('2個のサイコロ（|Ω|=36）における様々な事象（青＝事象に含まれる根元事象）',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('各事象の要素数と確率：')
for name, cond in events_def.items():
    event_set = [(d1, d2) for d1, d2 in omega2 if cond(d1, d2)]
    label = name.split('\n')[0]
    print(f'  {label}: |A|={len(event_set):2d}  P(A)={len(event_set)/36:.4f}')

---
## 4. 事象の演算

事象は集合であるため、集合演算がそのまま適用できます。

| 演算 | 記号 | 意味 | 集合表現 |
|------|------|------|----------|
| 和事象 | $A \cup B$ | A または B が起こる | 合併 |
| 積事象 | $A \cap B$ | A かつ B が起こる | 共通部分 |
| 補事象 | $A^c$ | A が起こらない | 補集合 |
| 差事象 | $A \setminus B$ | A は起こるが B は起こらない | 差集合 |

### 4-1. 和事象・積事象・補事象

In [ ]:
omega2 = [(d1, d2) for d1 in range(1, 7) for d2 in range(1, 7)]

# A: 和が7、B: ゾロ目
A_set = set((d1, d2) for d1, d2 in omega2 if d1 + d2 == 7)
B_set = set((d1, d2) for d1, d2 in omega2 if d1 == d2)

operations = [
    ('A\n（和が7）',       A_set,                      '#4C72B0'),
    ('B\n（ゾロ目）',       B_set,                      '#DD8452'),
    ('A ∪ B\n（和事象）',  A_set | B_set,               '#55A868'),
    ('A ∩ B\n（積事象）',  A_set & B_set,               '#C44E52'),
    ('A^c\n（補事象）',     set(omega2) - A_set,        '#8172B2'),
    ('A \ B\n（差事象）',  A_set - B_set,               '#937860'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, (name, event_set, color) in zip(axes, operations):
    for (d1, d2) in omega2:
        in_ev = (d1, d2) in event_set
        in_A  = (d1, d2) in A_set
        in_B  = (d1, d2) in B_set

        if in_ev:
            c = color
        elif in_A or in_B:
            c = '#E8E8E8'
        else:
            c = '#F5F5F5'

        circ = Circle((d1 - 1, d2 - 1), 0.4, color=c, zorder=2,
                      alpha=1.0 if in_ev else 0.6)
        ax.add_patch(circ)

        # Aの枠線
        if in_A:
            circ_a = Circle((d1 - 1, d2 - 1), 0.4,
                            fill=False, edgecolor='#4C72B0',
                            linewidth=1.5, zorder=3)
            ax.add_patch(circ_a)
        # Bの枠線
        if in_B:
            circ_b = Circle((d1 - 1, d2 - 1), 0.42,
                            fill=False, edgecolor='#DD8452',
                            linewidth=1.5, linestyle='dashed', zorder=4)
            ax.add_patch(circ_b)

        ax.text(d1 - 1, d2 - 1, f'{d1},{d2}',
                ha='center', va='center', fontsize=6,
                color='white' if in_ev else '#AAA',
                fontweight='bold' if in_ev else 'normal', zorder=5)

    ax.set_xlim(-0.6, 5.6)
    ax.set_ylim(-0.6, 5.6)
    ax.set_xticks(range(6))
    ax.set_xticklabels(range(1, 7), fontsize=8)
    ax.set_yticks(range(6))
    ax.set_yticklabels(range(1, 7), fontsize=8)
    ax.set_title(f'{name}\n|事象| = {len(event_set)}  P = {len(event_set)/36:.3f}',
                 fontsize=10)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

# 凡例
legend_elements = [
    mpatches.Patch(facecolor='white', edgecolor='#4C72B0', linewidth=1.5, label='A（和が7）の境界'),
    mpatches.Patch(facecolor='white', edgecolor='#DD8452', linewidth=1.5,
                   linestyle='dashed', label='B（ゾロ目）の境界'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.02))

fig.suptitle('事象の演算（A: 和が7、B: ゾロ目）\n色付き = 演算結果の事象',
             fontsize=13)
plt.tight_layout()
plt.show()

print('事象の演算結果：')
print(f'  |A|        = {len(A_set):2d}  P(A)       = {len(A_set)/36:.4f}')
print(f'  |B|        = {len(B_set):2d}  P(B)       = {len(B_set)/36:.4f}')
print(f'  |A ∪ B|   = {len(A_set|B_set):2d}  P(A∪B)    = {len(A_set|B_set)/36:.4f}')
print(f'  |A ∩ B|   = {len(A_set&B_set):2d}  P(A∩B)    = {len(A_set&B_set)/36:.4f}')
print(f'  |A^c|      = {len(set(omega2)-A_set):2d}  P(A^c)     = {len(set(omega2)-A_set)/36:.4f}')
print(f'  |A \ B|   = {len(A_set-B_set):2d}  P(A\\B)    = {len(A_set-B_set)/36:.4f}')
print()
print('確認: 包除原理 P(A∪B) = P(A) + P(B) - P(A∩B)')
print(f'  {len(A_set)/36:.4f} + {len(B_set)/36:.4f} - {len(A_set&B_set)/36:.4f}'
      f' = {(len(A_set)+len(B_set)-len(A_set&B_set))/36:.4f}'
      f' = {len(A_set|B_set)/36:.4f} ✓')

### 4-2. ベン図による可視化

事象の関係をベン図で直観的に確認します。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# ① A と B（和が7 vs ゾロ目）—— 交差あり
v1 = venn2(
    subsets=(len(A_set - B_set), len(B_set - A_set), len(A_set & B_set)),
    set_labels=('A: 和が7', 'B: ゾロ目'),
    ax=axes[0], set_colors=('#4C72B0', '#DD8452'), alpha=0.55
)
axes[0].set_title('AとBの関係（交差あり）\nA∩B = {(3,3)→和6… ※(1,6)等は和7}',
                  fontsize=9)

# ② C と D（1個目<=3 vs 和>=10）—— 一部交差
C_set = set((d1, d2) for d1, d2 in omega2 if d1 <= 3)
D_set = set((d1, d2) for d1, d2 in omega2 if d1 + d2 >= 10)
v2 = venn2(
    subsets=(len(C_set - D_set), len(D_set - C_set), len(C_set & D_set)),
    set_labels=('C: 1個目≤3', 'D: 和≥10'),
    ax=axes[1], set_colors=('#55A868', '#C44E52'), alpha=0.55
)
axes[1].set_title('CとDの関係\nC∩D は小さい（1個目が小さいと和は大きくなりにくい）',
                  fontsize=9)

# ③ 3事象のベン図
E_set = set((d1, d2) for d1, d2 in omega2 if d2 % 2 == 0)  # 2個目偶数
v3 = venn3(
    subsets=(
        len(A_set - B_set - E_set),
        len(B_set - A_set - E_set),
        len(A_set & B_set - E_set),
        len(E_set - A_set - B_set),
        len(A_set & E_set - B_set),
        len(B_set & E_set - A_set),
        len(A_set & B_set & E_set),
    ),
    set_labels=('A: 和が7', 'B: ゾロ目', 'E: 2個目偶数'),
    ax=axes[2], set_colors=('#4C72B0', '#DD8452', '#8172B2'), alpha=0.45
)
axes[2].set_title('3事象のベン図\nA, B, E の関係', fontsize=9)

fig.suptitle('事象のベン図による可視化', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'C (1個目<=3):   |C|={len(C_set)}  P(C)={len(C_set)/36:.4f}')
print(f'D (和>=10):     |D|={len(D_set)}  P(D)={len(D_set)/36:.4f}')
print(f'C∩D:           |C∩D|={len(C_set&D_set)}  P(C∩D)={len(C_set&D_set)/36:.4f}')
print(f'E (2個目偶数): |E|={len(E_set)}  P(E)={len(E_set)/36:.4f}')

### 4-3. 排反事象と完全加法系

**排反事象**（Mutually Exclusive Events）：$A \cap B = \emptyset$

**完全加法系**（Partition）：$\Omega$ を排反な事象に分割すること：
$$A_1, A_2, \ldots, A_k \text{ が互いに排反かつ } A_1 \cup A_2 \cup \cdots \cup A_k = \Omega$$

2個のサイコロの和 $S = d_1 + d_2 \in \{2,3,\ldots,12\}$ による $\Omega$ の分割を可視化します。

In [ ]:
# 和による標本空間の分割
sums = range(2, 13)
partition = {s: [(d1, d2) for d1, d2 in omega2 if d1 + d2 == s] for s in sums}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：グリッド上での色分け
ax = axes[0]
cmap = plt.cm.tab20
colors_part = {s: cmap((s - 2) / 10) for s in sums}

for (d1, d2) in omega2:
    s = d1 + d2
    circ = Circle((d1 - 1, d2 - 1), 0.4, color=colors_part[s], zorder=2)
    ax.add_patch(circ)
    ax.text(d1 - 1, d2 - 1, str(s),
            ha='center', va='center', fontsize=9,
            color='white', fontweight='bold', zorder=3)

ax.set_xlim(-0.6, 5.6)
ax.set_ylim(-0.6, 5.6)
ax.set_xticks(range(6))
ax.set_xticklabels(range(1, 7))
ax.set_yticks(range(6))
ax.set_yticklabels(range(1, 7))
ax.set_title('和 S = d1+d2 による Ω の分割\n（各色が排反事象 A_s = {和がs}）', fontsize=11)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

# 右：分割の確率分布
ax2 = axes[1]
counts = [len(partition[s]) for s in sums]
probs  = [c / 36 for c in counts]
bar_colors = [colors_part[s] for s in sums]
bars = ax2.bar(list(sums), probs, color=bar_colors, edgecolor='white', linewidth=1.2)
for bar, p, c in zip(bars, probs, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{c}/36', ha='center', fontsize=8)
ax2.set_xlabel('和 S', fontsize=12)
ax2.set_ylabel('P(A_s)', fontsize=12)
ax2.set_title(f'各排反事象の確率\n（合計 = {sum(probs):.4f} = 1）', fontsize=11)
ax2.set_xticks(list(sums))

fig.suptitle('完全加法系（Partition）：和による Ω の分割', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('和による分割（完全加法系の確認）：')
total = 0
for s in sums:
    p = len(partition[s]) / 36
    total += p
    print(f'  A_{s:2d}: {str(partition[s]):55s} |A|={len(partition[s]):2d}  P={p:.4f}')
print(f'  合計 P(Ω) = {total:.4f} ✓')

---
## 5. 直積空間と多段試行

**直積空間**（Product Space）：複数の独立した試行を組み合わせた標本空間。

$$\Omega = \Omega_1 \times \Omega_2 \times \cdots \times \Omega_n$$

「コイン1回 × サイコロ1個」の試行では $\Omega = \{H,T\} \times \{1,2,3,4,5,6\}$，$|\Omega|=12$。

In [ ]:
# コイン × サイコロ の直積空間
omega_coin = ['H', 'T']
omega_dice = list(range(1, 7))
omega_prod = list(product(omega_coin, omega_dice))

# いくつかの事象
ev_H     = {o for o in omega_prod if o[0] == 'H'}          # 表が出る
ev_even  = {o for o in omega_prod if o[1] % 2 == 0}        # サイコロ偶数
ev_H_even= ev_H & ev_even                                   # 両方

fig, ax = plt.subplots(figsize=(10, 4))

coin_y = {'H': 1, 'T': 0}
coin_colors = {'H': '#4C72B0', 'T': '#DD8452'}
event_colors = {
    'H_even':   '#55A868',
    'H_odd':    '#4C72B0',
    'T_even':   '#C44E52',
    'T_odd':    '#DD8452',
}

for (c, d) in omega_prod:
    x = d - 1
    y = coin_y[c]
    key = f'{c}_{"even" if d%2==0 else "odd"}'
    color = event_colors[key]
    circ = Circle((x, y), 0.35, color=color, zorder=2)
    ax.add_patch(circ)
    ax.text(x, y, f'({c},{d})',
            ha='center', va='center', fontsize=8.5,
            color='white', fontweight='bold', zorder=3)

# 全体の枠
rect = Rectangle((-0.5, -0.5), 6, 2,
                 linewidth=2, edgecolor='#555', facecolor='#f8f8f8', zorder=1)
ax.add_patch(rect)
ax.text(5.3, 1.3, 'Ω', fontsize=14, color='#555')

ax.set_xlim(-0.6, 5.7)
ax.set_ylim(-0.6, 1.7)
ax.set_xticks(range(6))
ax.set_xticklabels([f'サイコロ={i}' for i in range(1, 7)], fontsize=9)
ax.set_yticks([0, 1])
ax.set_yticklabels(['T（裏）', 'H（表）'], fontsize=10)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

legend_elements = [
    mpatches.Patch(color='#55A868', label='H かつ偶数'),
    mpatches.Patch(color='#4C72B0', label='H かつ奇数'),
    mpatches.Patch(color='#C44E52', label='T かつ偶数'),
    mpatches.Patch(color='#DD8452', label='T かつ奇数'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
          bbox_to_anchor=(1.0, 1.35))

ax.set_title(f'直積空間 Ω = {{H,T}} × {{1,2,3,4,5,6}}\n|Ω| = 2 × 6 = {len(omega_prod)}',
             fontsize=12)
plt.tight_layout()
plt.show()

print(f'Ω = {{H,T}} × {{1,...,6}}：|Ω| = {len(omega_prod)}')
print(f'  A（表が出る）:           |A|={len(ev_H)}     P(A)={len(ev_H)/len(omega_prod):.4f}')
print(f'  B（サイコロが偶数）:     |B|={len(ev_even)}     P(B)={len(ev_even)/len(omega_prod):.4f}')
print(f'  A∩B（表かつ偶数）:      |A∩B|={len(ev_H_even)}  P(A∩B)={len(ev_H_even)/len(omega_prod):.4f}')
print(f'  確認 P(A)×P(B) = {len(ev_H)/len(omega_prod):.4f} × {len(ev_even)/len(omega_prod):.4f}'
      f' = {len(ev_H)/len(omega_prod)*len(ev_even)/len(omega_prod):.4f} = P(A∩B) ✓（独立）')

---
## 6. 連続標本空間（区間・平面）

標本空間は離散的である必要はありません。連続標本空間では事象は区間・領域で表されます。

**例1**：バスが 0〜60 分の間にランダムに来る → $\Omega = [0, 60]$

**例2**：点が正方形 $[0,1]^2$ 上にランダムに落ちる → $\Omega = [0,1]^2$（幾何確率）

In [ ]:
np.random.seed(0)
n_pts = 3000

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── 例1：1次元連続標本空間（バス待ち時間）──────────────────────
ax = axes[0]
times = np.random.uniform(0, 60, n_pts)

# 事象A：10分以内に来る
A_mask = times <= 10
ax.hist(times[~A_mask], bins=30, range=(0,60), color='#DDDDDD',
        alpha=0.7, label='Ac（10分超）')
ax.hist(times[A_mask],  bins=5,  range=(0,10), color='#4C72B0',
        alpha=0.9, label='A（10分以内）')
ax.axvline(10, color='crimson', linestyle='--', linewidth=1.5)
ax.set_xlabel('待ち時間（分）', fontsize=11)
ax.set_ylabel('頻度', fontsize=11)
ax.set_title('Ω = [0, 60]（バス待ち時間）\nA = [0, 10]：P(A) = 10/60 ≈ 0.167',
             fontsize=10)
ax.legend(fontsize=9)

# ── 例2：2次元連続標本空間（幾何確率）────────────────────────
ax2 = axes[1]
pts = np.random.uniform(0, 1, (n_pts, 2))

# 事象B：単位円の内部 x^2+y^2 <= (0.5)^2
in_circle = (pts[:,0]-0.5)**2 + (pts[:,1]-0.5)**2 <= 0.25

ax2.scatter(pts[~in_circle, 0], pts[~in_circle, 1],
            s=2, color='#DDDDDD', alpha=0.6, label='Bc')
ax2.scatter(pts[in_circle, 0],  pts[in_circle, 1],
            s=2, color='#4C72B0', alpha=0.8, label='B（円内部）')
theta = np.linspace(0, 2*np.pi, 300)
ax2.plot(0.5 + 0.5*np.cos(theta), 0.5 + 0.5*np.sin(theta),
         'r--', linewidth=1.5)
ax2.set_aspect('equal')
ax2.set_title(f'Ω = [0,1]² （幾何確率）\nB = 円内部：P(B) = π/4 ≈ {np.pi/4:.4f}\n'
              f'モンテカルロ推定 = {in_circle.mean():.4f}', fontsize=10)
ax2.legend(fontsize=9, markerscale=4)
ax2.set_xlabel('x', fontsize=11)
ax2.set_ylabel('y', fontsize=11)

# ── 例3：π の推定（幾何確率の応用）──────────────────────────
ax3 = axes[2]
ns_pi = np.logspace(1, np.log10(n_pts), 200).astype(int)
pi_est = [4 * ((pts[:k, 0]-0.5)**2 + (pts[:k, 1]-0.5)**2 <= 0.25).mean()
          for k in ns_pi]

ax3.semilogx(ns_pi, pi_est, color='#4C72B0', linewidth=1.5)
ax3.axhline(np.pi, color='crimson', linestyle='--', linewidth=1.5,
            label=f'π = {np.pi:.5f}')
ax3.fill_between(ns_pi, np.pi - 0.05, np.pi + 0.05,
                 alpha=0.1, color='crimson')
ax3.set_xlabel('投下点数 n（対数スケール）', fontsize=11)
ax3.set_ylabel('4 × P(B) の推定値', fontsize=11)
ax3.set_title('幾何確率によるπの推定\n（P(B) = π/4 → 4P(B) → π）', fontsize=10)
ax3.legend(fontsize=10)

fig.suptitle('連続標本空間における事象', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'幾何確率によるπの推定値  = {4*in_circle.mean():.6f}')
print(f'真の値 π                = {np.pi:.6f}')
print(f'誤差                    = {abs(4*in_circle.mean()-np.pi):.6f}')

---
## 7. 事象の独立性と排反性の違い

初学者が混同しやすい重要な概念を比較します。

| 性質 | 定義 | ベン図の特徴 | $P(A \cap B)$ との関係 |
|------|------|------|------|
| **排反**（Mutually Exclusive） | $A \cap B = \emptyset$ | 重なりなし | $P(A \cap B) = 0$ |
| **独立**（Independent） | $P(A \cap B) = P(A)P(B)$ | 重なりあり（一般） | $P(A \cap B) = P(A)P(B)$ |

> **重要**：$P(A) > 0$ かつ $P(B) > 0$ のとき、排反と独立は**同時には成立しない**。
> 排反ならば $P(A \cap B) = 0 \neq P(A)P(B)$（∵ 右辺 > 0）。

In [ ]:
omega2 = [(d1, d2) for d1 in range(1, 7) for d2 in range(1, 7)]
n = len(omega2)

# ── ケース1：排反だが独立でない ──────────────────────────────
# A: 和が偶数、B: 和が奇数
A1 = set((d1,d2) for d1,d2 in omega2 if (d1+d2)%2==0)
B1 = set((d1,d2) for d1,d2 in omega2 if (d1+d2)%2==1)

# ── ケース2：独立だが排反でない ──────────────────────────────
# A: 1個目が偶数、B: 2個目が偶数（独立）
A2 = set((d1,d2) for d1,d2 in omega2 if d1%2==0)
B2 = set((d1,d2) for d1,d2 in omega2 if d2%2==0)

# ── ケース3：独立でも排反でもない ────────────────────────────
# A: 和が7、B: 2個目が3以下
A3 = set((d1,d2) for d1,d2 in omega2 if d1+d2==7)
B3 = set((d1,d2) for d1,d2 in omega2 if d2<=3)

cases = [
    ('排反・非独立\nA: 和偶数, B: 和奇数', A1, B1),
    ('独立・非排反\nA: 1個目偶数, B: 2個目偶数', A2, B2),
    ('非独立・非排反\nA: 和が7, B: 2個目<=3', A3, B3),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, (title, A, B) in zip(axes, cases):
    pA   = len(A) / n
    pB   = len(B) / n
    pAB  = len(A & B) / n
    pApB = pA * pB
    is_exclusive   = len(A & B) == 0
    is_independent = abs(pAB - pApB) < 1e-9

    # ベン図
    venn2(
        subsets=(len(A-B), len(B-A), len(A&B)),
        set_labels=('A', 'B'),
        ax=ax, set_colors=('#4C72B0','#DD8452'), alpha=0.5
    )

    status = []
    status.append('排反: ✓' if is_exclusive else 'A∩B ≠ ∅: 非排反')
    status.append('独立: ✓' if is_independent else f'P(A∩B)={pAB:.3f}≠P(A)P(B)={pApB:.3f}: 非独立')

    ax.set_title(
        f'{title}\n'
        f'P(A)={pA:.3f}, P(B)={pB:.3f}\n'
        f'P(A∩B)={pAB:.3f}, P(A)P(B)={pApB:.3f}\n'
        + '\n'.join(status),
        fontsize=9
    )

fig.suptitle('独立性 vs 排反性の違い（2個のサイコロ）', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('=== 独立性と排反性の確認 ===')
for title, A, B in cases:
    pA = len(A)/n; pB = len(B)/n; pAB = len(A&B)/n
    label = title.split('\n')[0]
    print(f'\n{label}')
    print(f'  P(A)={pA:.4f}, P(B)={pB:.4f}')
    print(f'  P(A∩B)   = {pAB:.4f}')
    print(f'  P(A)×P(B)= {pA*pB:.4f}')
    print(f'  排反: {len(A&B)==0}  独立: {abs(pAB-pA*pB)<1e-9}')

---
## まとめ

| 概念 | 定義 | 確認したセル |
|------|------|------|
| 標本空間 $\Omega$ | すべての根元事象の集合 | 2-1, 2-2 |
| 根元事象 | $\omega \in \Omega$（1点） | 2-1, 2-2 |
| 事象 | $A \subseteq \Omega$（部分集合） | 3 |
| べき集合 | $2^\Omega$（事象全体） | 2-3 |
| 和事象 | $A \cup B$ | 4-1 |
| 積事象 | $A \cap B$ | 4-1 |
| 補事象 | $A^c = \Omega \setminus A$ | 4-1 |
| 排反事象 | $A \cap B = \emptyset$ | 4-3 |
| 完全加法系 | $\Omega$ の排反分割 | 4-3 |
| 直積空間 | $\Omega_1 \times \Omega_2$ | 5 |
| 連続標本空間 | $\Omega = [a,b]$, $[0,1]^2$ 等 | 6 |
| 独立性 | $P(A \cap B) = P(A)P(B)$ | 7 |

> **本ノートブックと「確率の公理と性質」の関係**：
> 本ノートブックで構成した確率空間 $(\Omega, 2^\Omega, P)$ の上に、
> コルモゴロフの公理（非負性・全確率・加法性）を課すことで、
> 厳密な確率論の体系が構築されます。